# GitHub → Colab / VS Code → Google Drive

代码来自 `https://github.com/BoldHu/CellQuant.git` 的 `master` 分支，运行目录为 `/content/CellQuant`。
在本地 VS Code 编辑并 push；云端停下作业后 pull，重启 kernel 再运行。
完整步骤见 [工作流说明](../docs/COLAB_VSCODE.md)。

VS Code 先选择 **Select Kernel → Colab**，然后从上到下运行。本 notebook 已直接包含挂载 Google Drive 的单元，运行时按提示完成授权。
首次运行无需手动创建云盘中的 `CellQuant` 文件夹，挂载后会自动创建。`/content/drive/MyDrive` 是挂载入口；如果报它不存在，应先完成挂载，不能通过新建同名本地目录代替。
浏览器 Colab 使用同一个挂载单元。模型、数据、运行记录以版本化快照存到 Drive，实际计算读取本地盘。
安装依赖、G0 与备份由下面的配置控制；不会自动下载模型或启动真实实验。

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/BoldHu/CellQuant.git"
BRANCH = "master"
IN_COLAB = Path("/content").is_dir()
PROJECT_ROOT = Path(os.environ.get("CELLQUANT_PROJECT_ROOT", "/content/CellQuant" if IN_COLAB else str(Path.cwd())))
# 可修改为自己的 Drive 项目目录；运行下方挂载单元完成授权。
DRIVE_ROOT = os.environ.get("CELLQUANT_DRIVE_ROOT", "/content/drive/MyDrive/CellQuant") if IN_COLAB else None
DRIVE_ROOT = "/content/drive/MyDrive/CellQuant"
MOUNT_DRIVE = IN_COLAB  # 浏览器和 VS Code 的 Colab 内核都会挂载
INSTALL_BASE_DEPENDENCIES = True
RESTORE_AREAS = ["runs"]  # 新 runtime 需要恢复时填 ["models", "datasets", "runs"]
RESTORE_RUNS_CONFLICT = "separate"  # runs 冲突时将 Drive 完整快照另存；设 error 则停止
RUN_G0 = False      # 首次验证时显式设 True，只运行 CPU 数学检查
BACKUP_AREAS = []   # 完成阶段后填 ["runs"]；下载完成后另备份 models、datasets


## 挂载 Google Drive

运行下方单元，按 Google 弹窗完成登录授权，然后继续运行后续单元。
此命令同时用于浏览器 Colab 和 VS Code 的 Colab 内核；已挂载时会复用，非 Colab 环境默认跳过。
`CellQuant` 项目文件夹会在后续初始化中自动创建。


In [2]:
# 在 Colab（浏览器或 VS Code 内核）中直接挂载 Google Drive。
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive 挂载尚未完成，请完成授权后重跑本单元。")
    print("Google Drive 已挂载：/content/drive/MyDrive")
else:
    print("非 Colab 环境或 MOUNT_DRIVE=False：跳过云盘挂载。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 已挂载：/content/drive/MyDrive


In [3]:
PROJECT_ROOT = PROJECT_ROOT.resolve()
if IN_COLAB and not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the CellQuant checkout")
if IN_COLAB:
    origin = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different origin; choose a new PROJECT_ROOT")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["PYTHONUTF8"] = "1"
from cellquant.workspace import git_identity
print({"project": str(PROJECT_ROOT), "code": git_identity(PROJECT_ROOT)})


{'project': '/content/CellQuant', 'code': {'commit': '8eb21e72a3396c7462ac45917553f3711a21cae6', 'branch': 'master', 'dirty': False}}


In [4]:
from cellquant.workspace import configure_workspace
WORKSPACE_READY = False
if DRIVE_ROOT:
    record = configure_workspace(PROJECT_ROOT, DRIVE_ROOT)
    WORKSPACE_READY = True
    print(record)
else:
    print("Offline workspace check only; Drive persistence NOT_RUN")


{'schema_version': '1.0', 'created_at_utc': '2026-11-16T08:11:59.493613+00:00', 'environment': {'CELLQUANT_PROJECT_ROOT': '/content/CellQuant', 'CELLQUANT_RUN_ROOT': '/content/CellQuant/runs', 'CELLQUANT_CACHE_DIR': '/content/CellQuant/.cache/huggingface/hub', 'HF_HOME': '/content/CellQuant/.cache/huggingface', 'HF_HUB_CACHE': '/content/CellQuant/.cache/huggingface/hub', 'HF_DATASETS_CACHE': '/content/CellQuant/.cache/huggingface/datasets', 'HF_TOKEN_PATH': '/content/CellQuant/.cache/huggingface/token', 'PYTHONUTF8': '1', 'CELLQUANT_DRIVE_ROOT': '/content/drive/MyDrive/CellQuant'}, 'code': {'commit': '8eb21e72a3396c7462ac45917553f3711a21cae6', 'branch': 'master', 'dirty': False}, 'source_type': 'not_run', 'status': 'CONFIGURED', 'note': 'No models downloaded, experiments run, or scientific gates passed'}


## 安装依赖与恢复快照

`RESTORE_AREAS` 只在需要取回 Drive 备份时设置；重启 kernel 后当前 VM 的文件仍可能存在。
若已有 `runs` 与备份内容不同，`RESTORE_RUNS_CONFLICT="separate"` 会保留现有文件，
将 Drive 中的**整个快照**恢复到项目根目录的 `restored_snapshots/runs/<快照哈希>/`，并输出 `RESTORED_SEPARATELY` 和 `destination`。
这是历史副本，不会切换当前运行目录、不重写配置、不授予旧 checkpoint 或 gate 续跑资格。
没有冲突时恢复到原路径；模型和数据缓存冲突仍会停止。请先停止所有会写这些目录的作业。


In [ ]:
from cellquant.runtime import stream_command
from datetime import datetime, timezone
import uuid

def logged_command(argv, label, timeout=21600):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
    # Backup logs live outside runs, so backup never copies its own changing log.
    log = PROJECT_ROOT / ".cache/workspace_logs" / f"{label}_{stamp}.log"
    result = stream_command(argv, log, cwd=PROJECT_ROOT, timeout_s=timeout)
    if result["returncode"]:
        with log.open("rb") as handle:
            handle.seek(max(0, log.stat().st_size - 4096))
            tail = handle.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{label} failed (exit {result['returncode']}); see {log}\n\n{tail}")
    return result

if INSTALL_BASE_DEPENDENCIES:
    logged_command([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], "install", 1800)
else:
    print("Dependency installation NOT_RUN; existing torch is preserved")
if RESTORE_AREAS:
    if not WORKSPACE_READY:
        raise RuntimeError("Configure mounted Drive first")
    restore_args = [sys.executable, "-m", "cellquant.workspace", "restore", "--areas", *RESTORE_AREAS]
    if "runs" in RESTORE_AREAS:
        restore_args += ["--runs-conflict", RESTORE_RUNS_CONFLICT]
    logged_command(restore_args, "restore")


In [ ]:
if RUN_G0:
    from cellquant.runtime import atomic_json, environment_report, read_json
    output = PROJECT_ROOT / "runs/00_math"
    atomic_json(output / "environment.json", environment_report())
    for script, subdir in [("run_math_checks.py", "reference"), ("run_production_checks.py", "production")]:
        logged_command([sys.executable, f"scripts/{script}", "--output", str(output / subdir)], subdir, 600)
    print("Measured G0:", read_json(output / "production/G0.json")["status"])
else:
    print("G0 execution NOT_RUN in this notebook invocation")

if BACKUP_AREAS:
    if not WORKSPACE_READY:
        raise RuntimeError("Configure mounted Drive first")
    logged_command([sys.executable, "-m", "cellquant.workspace", "backup", "--areas", *BACKUP_AREAS], "backup")


{
  "status": "PASS",
  "checks": {
    "orthogonality": true,
    "product_equivalence": true,
    "screening_event_equivalence": true,
    "cell_partition": true,
    "analytic_product": true,
    "analytic_loss": true,
    "analytic_budgets": true,
    "actual_acceptance": true,
    "independent_requantization": true,
    "rounding_and_zero": true
  },
  "report": "/content/CellQuant/runs/00_math/reference/math_reference_report.json"
}
G0: PASS
Measured G0: PASS
[
  {
    "status": "BACKED_UP",
    "area": "runs",
    "files": 5,
    "transferred_bytes": 67439,
    "snapshot": "/content/drive/MyDrive/CellQuant/snapshots/runs/20261116T075752803471Z_4ec2b9fe.json",
    "note": "Existing objects are hash-verified on restore"
  }
]


## 后续使用

完成 G0 后打开 `01_Colab_Real_Layer_Pilot.ipynb`，并按 `docs/RUNBOOK.md` 准备固定 SpinQuant 和真实数据。
同一服务器上的不同 notebook 可能有不同 kernel；现有阶段 notebook 会读取本地 `.cache/colab_workspace.json` 恢复路径设置。

下载模型/数据后，在无下载或训练写入的阶段边界运行：
```python
logged_command([sys.executable, "-m", "cellquant.workspace", "backup", "--areas", "models", "datasets", "runs"], "backup")
```
Drive 中 `objects/` 是按 SHA-256 去重的文件，`snapshots/` 是版本索引，两者须一同保留。中断传输不发布成功索引；备份不改变科学门槛。
新的 runtime 先 clone、挂载、设置路径，再恢复需要的区域。更新代码后旧结果保留，恢复实验仍必须核对代码/环境/数据指纹。

停下所有云端作业后，打开官方扩展的 Colab terminal，运行：
```bash
cd /content/CellQuant
git status --short
git pull --ff-only origin master
python -m pip install -e '.[dev]'
```
有本地改动时先自行保存并解决；不要强制 reset。随后重启 notebook kernel，并重跑初始化单元。
如果没有 terminal，可在云端 notebook 用 `subprocess.run([...], check=True)` 执行同样的 Git 命令。
